In [1]:
import requests
import pandas as pd

# API call - filter directly on Africa
url = "https://restcountries.com/v3.1/region/africa"
response = requests.get(url)
data = response.json()

print(f"✅ {len(data)} african countries recovered")

✅ 59 african countries recovered


In [2]:
# Cleaning & structuring
records = []

for country in data:
    records.append({
        "Country":      country.get("name", {}).get("common", "N/A"),
        "Capital":      country.get("capital", ["N/A"])[0] if country.get("capital") else "N/A",
        "Subregion":    country.get("subregion", "N/A"),
        "Population":   country.get("population", 0),
        "Area_km2":     country.get("area", 0),
        "Nb_Languages": len(country.get("languages", {})),
        "Nb_Currencies":len(country.get("currencies", {})),
        "Landlocked":   country.get("landlocked", False),
        "Nb_Borders":   len(country.get("borders", []))
    })

df_africa = pd.DataFrame(records)
df_africa = df_africa.sort_values("Population", ascending=False)

print(df_africa.head(10))
print(f"\nShape : {df_africa.shape}")

         Country      Capital        Subregion  Population   Area_km2  \
7        Nigeria        Abuja   Western Africa   223800000   923768.0   
53      DR Congo     Kinshasa    Middle Africa   112832000  2344858.0   
41      Ethiopia  Addis Ababa   Eastern Africa   111652998  1104300.0   
35         Egypt        Cairo  Northern Africa   107271260  1002450.0   
47      Tanzania       Dodoma   Eastern Africa    68153004   947303.0   
16  South Africa     Pretoria  Southern Africa    63100945  1221037.0   
56         Kenya      Nairobi   Eastern Africa    53330978   580367.0   
50         Sudan     Khartoum  Northern Africa    51662000  1886068.0   
8        Algeria      Algiers  Northern Africa    47400000  2381741.0   
29        Uganda      Kampala   Eastern Africa    45905417   241550.0   

    Nb_Languages  Nb_Currencies  Landlocked  Nb_Borders  
7              1              1       False           4  
53             5              1       False           9  
41             1      

In [3]:
# export
df_africa.to_csv("africa_countries_api.csv", index=False)
print("✅ Successful export : africa_countries_api.csv")

✅ Successful export : africa_countries_api.csv


In [2]:
# Load the dataset
import pandas as pd
df = pd.read_csv("africa_countries_api.csv")

# Display variable types
print(df.dtypes)

# Display shape
print(f"\nShape: {df.shape}")

# Preview first rows
print(df.head())

Country           object
Capital           object
Subregion         object
Population         int64
Area_km2         float64
Nb_Languages       int64
Nb_Currencies      int64
Landlocked          bool
Nb_Borders         int64
dtype: object

Shape: (59, 9)
    Country      Capital        Subregion  Population   Area_km2  \
0   Nigeria        Abuja   Western Africa   223800000   923768.0   
1  DR Congo     Kinshasa    Middle Africa   112832000  2344858.0   
2  Ethiopia  Addis Ababa   Eastern Africa   111652998  1104300.0   
3     Egypt        Cairo  Northern Africa   107271260  1002450.0   
4  Tanzania       Dodoma   Eastern Africa    68153004   947303.0   

   Nb_Languages  Nb_Currencies  Landlocked  Nb_Borders  
0             1              1       False           4  
1             5              1       False           9  
2             1              1        True           6  
3             1              1       False           4  
4             2              1       False         

In [4]:
# Check for missing values
print(df.isnull().sum())

Country          0
Capital          0
Subregion        0
Population       0
Area_km2         0
Nb_Languages     0
Nb_Currencies    0
Landlocked       0
Nb_Borders       0
dtype: int64


In [5]:
# Select numeric columns relevant to business question
numeric_cols = ["Population", "Area_km2", "Nb_Languages", "Nb_Currencies", "Nb_Borders"]

# Detect outliers using IQR method
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]["Country"].tolist()
    print(f"{col} — Outliers: {outliers}")

Population — Outliers: ['Nigeria', 'DR Congo', 'Ethiopia', 'Egypt']
Area_km2 — Outliers: ['DR Congo', 'Algeria']
Nb_Languages — Outliers: ['DR Congo', 'South Africa', 'Zimbabwe', 'Namibia']
Nb_Currencies — Outliers: ['Namibia', 'Lesotho', 'Eswatini', 'Western Sahara', 'Saint Helena, Ascension and Tristan da Cunha']
Nb_Borders — Outliers: []


In [6]:
# Check outlier impact: outliers here are REAL geographic/demographic facts
# (e.g. Nigeria has genuinely high population, Algeria has genuinely large area)
# No correction applied — outliers are valid data points, not data entry errors
print("\nNo corrections needed: no missing values detected.")
print("Outliers reflect real-world geographic and demographic diversity — retained as-is.")


No corrections needed: no missing values detected.
Outliers reflect real-world geographic and demographic diversity — retained as-is.


In [7]:
# Convert boolean to integer (0/1) — required for SQL TINYINT and Power BI compatibility
df["Landlocked"] = df["Landlocked"].astype(int)

# Confirm Area_km2 stays float — no rounding needed, already clean
df["Area_km2"] = df["Area_km2"].astype(float)

# Confirm Population as int64 — compatible with SQL BIGINT
df["Population"] = df["Population"].astype("int64")

# Confirm integer columns — compatible with SQL INT
int_cols = ["Nb_Languages", "Nb_Currencies", "Nb_Borders"]
for col in int_cols:
    df[col] = df[col].astype(int)

# Confirm string columns — compatible with SQL VARCHAR
str_cols = ["Country", "Capital", "Subregion"]
for col in str_cols:
    df[col] = df[col].astype(str)

# Verify final types
print(df.dtypes)

# Preview
print(df.head())

Country           object
Capital           object
Subregion         object
Population         int64
Area_km2         float64
Nb_Languages       int64
Nb_Currencies      int64
Landlocked         int64
Nb_Borders         int64
dtype: object
    Country      Capital        Subregion  Population   Area_km2  \
0   Nigeria        Abuja   Western Africa   223800000   923768.0   
1  DR Congo     Kinshasa    Middle Africa   112832000  2344858.0   
2  Ethiopia  Addis Ababa   Eastern Africa   111652998  1104300.0   
3     Egypt        Cairo  Northern Africa   107271260  1002450.0   
4  Tanzania       Dodoma   Eastern Africa    68153004   947303.0   

   Nb_Languages  Nb_Currencies  Landlocked  Nb_Borders  
0             1              1           0           4  
1             5              1           0           9  
2             1              1           1           6  
3             1              1           0           4  
4             2              1           0           8  


In [8]:
# Rename columns for business readability and SQL/Power BI compatibility
df = df.rename(columns={
    "Country"       : "country",
    "Capital"       : "capital_city",
    "Subregion"     : "african_region",
    "Population"    : "total_population",
    "Area_km2"      : "area_km2",
    "Nb_Languages"  : "language_count",
    "Nb_Currencies" : "currency_count",
    "Landlocked"    : "is_landlocked",
    "Nb_Borders"    : "border_count"
})

# Verify new column names
print(df.columns.tolist())

# Preview renamed dataset
print(df.head())

['country', 'capital_city', 'african_region', 'total_population', 'area_km2', 'language_count', 'currency_count', 'is_landlocked', 'border_count']
    country capital_city   african_region  total_population   area_km2  \
0   Nigeria        Abuja   Western Africa         223800000   923768.0   
1  DR Congo     Kinshasa    Middle Africa         112832000  2344858.0   
2  Ethiopia  Addis Ababa   Eastern Africa         111652998  1104300.0   
3     Egypt        Cairo  Northern Africa         107271260  1002450.0   
4  Tanzania       Dodoma   Eastern Africa          68153004   947303.0   

   language_count  currency_count  is_landlocked  border_count  
0               1               1              0             4  
1               5               1              0             9  
2               1               1              1             6  
3               1               1              0             4  
4               2               1              0             8  


In [9]:
# --- Variable 1: Population density (inhabitants per km2) ---
df["population_density"] = (df["total_population"] / df["area_km2"]).round(2)

# --- Variable 2: Population rank (1 = most populated country in Africa) ---
df["population_rank"] = df["total_population"].rank(ascending=False).astype(int)

# --- Variable 3: Area category based on quartiles ---
q1 = df["area_km2"].quantile(0.25)
q2 = df["area_km2"].quantile(0.50)
q3 = df["area_km2"].quantile(0.75)

df["area_category"] = pd.cut(
    df["area_km2"],
    bins=[0, q1, q2, q3, df["area_km2"].max()],
    labels=["Small", "Medium", "Large", "Very Large"]
)

# --- Variable 4: Linguistic diversity level based on business thresholds ---
df["linguistic_diversity_level"] = pd.cut(
    df["language_count"],
    bins=[0, 3, 6, 10, df["language_count"].max()],
    labels=["Low", "Medium", "High", "Very High"]
)

# --- Variable 5: Connectivity profile combining landlocked status and border count ---
def connectivity_profile(row):
    if row["is_landlocked"] == 1 and row["border_count"] <= 2:
        return "Isolated"
    elif row["is_landlocked"] == 1 and row["border_count"] >= 3:
        return "Landlocked Connected"
    elif row["is_landlocked"] == 0 and row["border_count"] <= 2:
        return "Coastal Low Connect"
    else:
        return "Coastal High Connect"

df["connectivity_profile"] = df.apply(connectivity_profile, axis=1)

# --- Variable 6: Complexity score (normalized 0-100 combining 3 dimensions) ---
# Normalize each dimension between 0 and 1 then scale to 100
pop_norm  = (df["total_population"] - df["total_population"].min()) / (df["total_population"].max() - df["total_population"].min())
area_norm = (df["area_km2"] - df["area_km2"].min()) / (df["area_km2"].max() - df["area_km2"].min())
lang_norm = (df["language_count"] - df["language_count"].min()) / (df["language_count"].max() - df["language_count"].min())

df["complexity_score"] = ((pop_norm + area_norm + lang_norm) / 3 * 100).round(2)

# Verify all new variables
print(df[["country", "population_density", "population_rank",
          "area_category", "linguistic_diversity_level",
          "connectivity_profile", "complexity_score"]].head(10))

# Final shape
print(f"\nDataset shape: {df.shape}")
print(df.dtypes)

        country  population_density  population_rank area_category  \
0       Nigeria              242.27                1    Very Large   
1      DR Congo               48.12                2    Very Large   
2      Ethiopia              101.11                3    Very Large   
3         Egypt              107.01                4    Very Large   
4      Tanzania               71.94                5    Very Large   
5  South Africa               51.68                6    Very Large   
6         Kenya               91.89                7         Large   
7         Sudan               27.39                8    Very Large   
8       Algeria               19.90                9    Very Large   
9        Uganda              190.05               10        Medium   

  linguistic_diversity_level  connectivity_profile  complexity_score  
0                        Low  Coastal High Connect             46.26  
1                     Medium  Coastal High Connect             59.15  
2               

In [10]:
# Drop capital_city — no KPI or Power BI visual uses it
df = df.drop(columns=["capital_city"])

# Verify final columns
print(df.columns.tolist())

# Verify final shape
print(f"\nFinal dataset shape: {df.shape}")

# Preview final dataset
print(df.head())

# Export clean final dataset to CSV for SQL import
df.to_csv("africa_countries_clean.csv", index=False)
print("\nClean dataset exported: africa_countries_clean.csv")

['country', 'african_region', 'total_population', 'area_km2', 'language_count', 'currency_count', 'is_landlocked', 'border_count', 'population_density', 'population_rank', 'area_category', 'linguistic_diversity_level', 'connectivity_profile', 'complexity_score']

Final dataset shape: (59, 14)
    country   african_region  total_population   area_km2  language_count  \
0   Nigeria   Western Africa         223800000   923768.0               1   
1  DR Congo    Middle Africa         112832000  2344858.0               5   
2  Ethiopia   Eastern Africa         111652998  1104300.0               1   
3     Egypt  Northern Africa         107271260  1002450.0               1   
4  Tanzania   Eastern Africa          68153004   947303.0               2   

   currency_count  is_landlocked  border_count  population_density  \
0               1              0             4              242.27   
1               1              0             9               48.12   
2               1              1 

In [11]:
# Convert to Excel
df.to_excel("africa_countries_clean.xlsx",index=False,sheet_name="africa_countries")

In [12]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

user = "root"
password = "mysql2026"
host = "localhost"
port = 3306
db = "esmel"

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{db}",
    echo=False
)

In [13]:
engine.connect()

In [14]:
df.to_sql(
    name="africa_countries",
    con=engine,
    if_exists="replace",
    index=False
)

59